# Part 1: Find Your Twin

**Today we learn** how a computer decides that two things are similar. **We do it on ourselves:** you describe yourself as numbers, then hunt for the classmate who matches you best.

**Example.** Ana and Ben both built a robot arm last month and both love having no classes. The computer scores them 92%. Ana and Chris did a cyber project and a game project, and feel differently about autonomy: 31%. By the end you will know exactly why.

**Your mission:** walk around the OIL, interview classmates, and find someone with a **95% match**. Write down every name and score. Stop at 95%, or after 10 people and report your best.

Time: about 60 minutes.

In [ ]:
!pip install -q numpy pandas scikit-learn matplotlib

## Describe yourself as numbers

Each person gives two sets of answers: one for your **past project** (last four weeks) and one for your **current project**.

| Feature | Type | Values |
|---|---|---|
| area | category | 0 cyber, 1 embedded, 2 game design, 3 AI, 4 business, 5 software |
| cyber, hw_robot, ai_ml, game, business | yes or no | 1 if the project involved it, else 0 |
| orientation_sure | scale | 0 = unsure, 1 = very sure about your orientation |
| no_classes | scale | 0 = hate having no classes, 1 = love it |
| autonomy | scale | 0 = dislike the autonomy, 1 = love it |

**Why numbers?** A computer cannot judge that two people are alike. It can only compare numbers, so we turn each answer into one, and "how alike" becomes a measurement instead of an opinion.

Edit the dictionary so it describes you.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics.pairwise import cosine_similarity

AREAS  = {0: "cyber", 1: "embedded", 2: "game design", 3: "AI", 4: "business", 5: "software"}
BINARY = ["cyber", "hw_robot", "ai_ml", "game", "business"]
SCALE  = ["orientation_sure", "no_classes", "autonomy"]

me = {
    "name": "YOUR NAME",
    "past_area": 1, "past_cyber": 0, "past_hw_robot": 1, "past_ai_ml": 0, "past_game": 0, "past_business": 0,
    "now_area": 3,  "now_cyber": 0,  "now_hw_robot": 0,  "now_ai_ml": 1,  "now_game": 0,  "now_business": 0,
    "orientation_sure": 0.6, "no_classes": 0.8, "autonomy": 0.7,
}

: 

## Check the data before you trust it

**Why check?** One typo, like area 9 or autonomy 7, silently ruins every score after it, and nothing will warn you. A few lines that reject bad rows protect the whole class from confusing results.

In [ ]:
def validate(row):
    assert row["name"].strip(), "name is empty"
    for p in ("past", "now"):
        assert row[f"{p}_area"] in AREAS, f"{p}_area must be one of {list(AREAS)}"
        for b in BINARY:
            assert row[f"{p}_{b}"] in (0, 1), f"{p}_{b} must be 0 or 1"
    for s in SCALE:
        assert 0 <= row[s] <= 1, f"{s} must be between 0 and 1"
    return row

validate(me)
print("Your row is valid.")

## See yourself as squares

**Why draw it?** Your eyes spot a good match faster than a table of numbers does. Two rows of squares that look alike should score high later. If they do not, something is off.

In [ ]:
def pattern(row):                      # 25 squares, each on (1) or off (0)
    bits = []
    for p in ("past", "now"):
        bits += [int(row[f"{p}_area"] == k) for k in AREAS]    # 6 squares: which area
        bits += [row[f"{p}_{b}"] for b in BINARY]              # 5 squares: yes or no
    bits += [int(row[s] >= 0.5) for s in SCALE]                # 3 squares: feeling above 0.5?
    return bits

def show(rows):
    fig, ax = plt.subplots(figsize=(9, 1 + 0.6 * len(rows)))
    ax.imshow([pattern(r) for r in rows], cmap="Greys", aspect="equal")
    ax.set_yticks(range(len(rows)))
    ax.set_yticklabels([r["name"] for r in rows])
    ax.set_xticks([2.5, 8, 13.5, 19, 23.5])
    ax.set_xticklabels(["past area", "past yes/no", "now area", "now yes/no", "feelings"], fontsize=7)
    plt.show()

show([me])

## Collect classmates

**Why collect by hand?** Every match is only as good as the answers behind it. Typing the data yourself shows how fragile that is, and why the checks above matter.

First practice with five made-up people (`USE_DEMO = True`). When you are ready, set it to `False` and interview real classmates. Write down their names.

In [ ]:
def interview():
    row = {"name": input("Name: ").strip()}
    for p in ("past", "now"):
        row[f"{p}_area"] = int(input(f"{p} area {AREAS}: "))
        for b in BINARY:
            row[f"{p}_{b}"] = int(input(f"{p}: involved {b}? (0/1): "))
    for s in SCALE:
        row[s] = float(input(f"{s} (0 to 1): "))
    return validate(row)

def fake(name, past_area, past_flags, now_area, now_flags, feelings):
    row = {"name": name, "past_area": past_area, "now_area": now_area}
    row.update({f"past_{b}": v for b, v in zip(BINARY, past_flags)})
    row.update({f"now_{b}": v for b, v in zip(BINARY, now_flags)})
    row.update(dict(zip(SCALE, feelings)))
    return row

demo = [
    fake("Demo Ana",   1, [0,1,0,0,0], 1, [0,1,0,0,0], [0.8, 0.9, 0.7]),
    fake("Demo Ben",   1, [0,1,0,0,0], 3, [0,0,1,0,0], [0.7, 0.8, 0.8]),
    fake("Demo Chris", 0, [1,0,0,0,0], 0, [1,0,0,0,0], [0.2, 0.3, 0.4]),
    fake("Demo Dana",  2, [0,0,0,1,0], 2, [0,0,0,1,0], [0.9, 0.5, 0.9]),
    fake("Demo Eli",   4, [0,0,0,0,1], 5, [0,0,1,0,0], [0.4, 0.6, 0.5]),
]

USE_DEMO = True
people = [validate(me)] + (demo if USE_DEMO else [])

In [ ]:
# Run this cell, then answer the questions for each real classmate. Type n to stop.
while input("Interview someone? y/n: ").strip().lower() == "y":
    people.append(interview())

df = pd.DataFrame(people)
df.to_csv("class_data.csv", index=False)      # saved, so a crash does not cost you your interviews
df[["name", "past_area", "now_area"]]

## Prepare the features

**Why three steps?** Each step stops the computer from misreading you. One-hot encoding stops it from inventing an order between areas, centering makes "dislike" and "like" point in opposite directions, and truncating removes digits nobody really meant. The comments in the code say which is which.

Fill in the one missing line.

In [ ]:
def trunc2(x):
    # Keep 2 decimals by cutting off the rest: 0.5173 becomes 0.51.
    # People's "0.7 or 0.73?" answers are guesses, so extra digits are fake precision.
    # The 1e-6 stops float errors, which would turn 0.29 into 0.28.
    return np.floor(np.asarray(x, float) * 100 + 1e-6) / 100

def preprocess(df, phase="past"):
    # One-hot the area. Codes 0 to 5 are labels, not quantities: raw numbers would make
    # "software (5)" look far from "cyber (0)". Fixed categories give everyone the same columns.
    area  = OneHotEncoder(categories=[list(AREAS)], sparse_output=False).fit_transform(df[[f"{phase}_area"]])
    yesno = df[[f"{phase}_{b}" for b in BINARY]].to_numpy()
    # Truncate first, then center at 0.5. Cosine only compares direction, so raw (0.1, 0.1, 0.1)
    # and (0.9, 0.9, 0.9) would look identical. Centering makes dislike negative and like positive.
    feelings = trunc2(df[SCALE].to_numpy()) - 0.5

    # TODO: glue the three blocks side by side into one matrix X (hint: np.hstack)
    X = None

    assert not np.isnan(X).any(), "missing values"
    assert (np.linalg.norm(X, axis=1) > 0).all(), "a row of zeros has no direction"
    return X

<details><summary>Solution</summary>

```python
X = np.hstack([area, yesno, feelings])
```

</details>

In [ ]:
X = preprocess(df)
print(X.shape, "= people x features")     # 6 area + 5 yes/no + 3 feelings = 14 columns
X.round(2)[:3]

## Compute your match

**Why cosine similarity?** It compares the *direction* of two lists of numbers and ignores their size, which is exactly what we want when asking "do we answer the same way?" Same direction scores 100%, unrelated scores 0%, opposite scores below 0.

**ELI5.** Two friends point at things. Same direction, they agree. At right angles, nothing in common. Opposite, they disagree. Play with the arrows, then run the match.

In [ ]:
from IPython.display import HTML

HTML("""<div style="font-family:system-ui,sans-serif;max-width:440px;margin:auto">
<label>Arrow A angle: <b id="cd_la">20</b>&deg;</label>
<input id="cd_sa" type="range" min="0" max="360" value="20" style="width:100%">
<label>Arrow B angle: <b id="cd_lb">70</b>&deg;</label>
<input id="cd_sb" type="range" min="0" max="360" value="70" style="width:100%">
<svg viewBox="-110 -110 220 220" width="220" height="220" style="display:block;margin:auto">
<circle r="90" fill="none" stroke="gray" stroke-opacity=".5"/>
<line id="cd_A" x1="0" y1="0" stroke="#d4561f" stroke-width="4" stroke-linecap="round"/>
<line id="cd_B" x1="0" y1="0" stroke="#2f5bea" stroke-width="4" stroke-linecap="round"/></svg>
<p style="text-align:center;margin:4px">cosine similarity = <b id="cd_cs" style="font-size:1.6em">0.64</b> &nbsp; <span id="cd_pct">64% match</span></p>
<p id="cd_msg" style="text-align:center;margin:4px;opacity:.8"></p></div>
<script>
(function(){var $=function(i){return document.getElementById(i)};
function u(){var a=+$("cd_sa").value,b=+$("cd_sb").value,r=Math.PI/180,c=Math.cos((a-b)*r);
$("cd_la").textContent=a;$("cd_lb").textContent=b;
$("cd_A").setAttribute("x2",90*Math.cos(a*r));$("cd_A").setAttribute("y2",-90*Math.sin(a*r));
$("cd_B").setAttribute("x2",90*Math.cos(b*r));$("cd_B").setAttribute("y2",-90*Math.sin(b*r));
$("cd_cs").textContent=c.toFixed(2);$("cd_pct").textContent=Math.round(c*100)+"% match";
$("cd_msg").textContent=c>=0.95?"Almost the same direction: twins!":c>0.5?"Pointing roughly the same way":c>-0.5?"Not much in common":"Pointing in opposite directions";}
$("cd_sa").oninput=$("cd_sb").oninput=u;u();})();
</script>""")

In [ ]:
def match_report(df, my_name, phase="past", target=0.95):
    sims = cosine_similarity(preprocess(df, phase))      # every person against every person
    i = df.index[df["name"] == my_name][0]
    # TODO: go through the other people from best match to worst.
    #   Print "name: xx%" using sims[i, j] * 100, and add TWIN FOUND when the score is at least target.
    #   Hint: np.argsort(-sims[i]) lists people from highest score to lowest, and skip yourself (j == i).
    pass

match_report(df, me["name"])

<details><summary>Solution</summary>

```python
for j in np.argsort(-sims[i]):
    if j == i: continue
    s = sims[i, j]
    print(f"{df.loc[j, 'name']}: {s * 100:.0f}%", "  TWIN FOUND" if s >= target else "")
```

</details>

In [ ]:
# Compare your squares with your best match.
twin_name = "Demo Ana"          # type the best name from the list above
show([me, df[df["name"] == twin_name].iloc[0].to_dict()])

Try `match_report(df, me["name"], phase="now")`. Is your twin the same person?

## Questions to discuss

**Why these questions?** The method works, but it hides choices about what to ask and how to encode it. These questions make the hidden choices visible.

1. Everyone answered the **same** 25 questions. If each person had their **own** features, how would you find a match?
2. What is useful about describing people as vectors? What does it leave out? Who gets a 95% match by accident?
3. **Can we turn people into vectors?** Which parts of you survive the conversion, and which do not?
4. What is **your** most unique feature, one none of the questions captured? Would you encode it as yes or no, a category, a 0 to 1 scale, or free text? Add it as a column and see whose score changes.

## Appendix: definitions and reasons

| Term | What it is | Why we care |
|---|---|---|
| Vector | An ordered list of numbers | Lets a computer do geometry on anything |
| Cosine similarity | The angle between two vectors, from 1 to -1 | Compares direction and ignores size |
| Categorical variable | A choice among labels, like the project area | Codes such as 0 to 5 have no real order |
| Binary variable | Only 0 or 1, a yes or no | Already numeric, needs no conversion |
| Scale variable | A number in a range, like 0 to 1 | Needs the same direction in every question |
| One-hot encoding | One column per category, 1 for the chosen one | Stops the computer inventing an order |
| Centering | Subtracting a middle value, here 0.5 | Makes opposite feelings point opposite ways |
| Truncation | Cutting digits after the second decimal | Removes fake precision from human answers |
| Validation | Checking data before using it | One bad entry corrupts every score |